In [1]:
import pandas as pd
import transformers
import torch
import re
import os
from tqdm import tqdm
from transformers import AutoTokenizer

# 1. 환경 설정 및 모델 로딩 (이전과 동일)
# ---------------------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
model_id = "tiiuae/falcon-40b-instruct"

print("Tokenizer 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Pipeline 설정 중... (시간이 소요될 수 있습니다)")
pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
)
print("모델 및 Pipeline 준비 완료.")


# 2. 프롬프트 템플릿 정의 (이전과 동일)
# ---------------------------------------------------
prompt_template = """
### ROLE
You are an AI assistant specializing in creating rich, descriptive educational content. Your main task is to elaborate on simple multiple-choice options to make them more informative and engaging.

### INSTRUCTION
I will provide you with a [QUESTION] and four short [OPTIONS]. Your job is to rewrite each option to be a longer, more descriptive sentence or phrase. Follow these rules:
1.  Do not change the core meaning of the original option.
2.  Add vivid details, context, or relevant information to each option.
3.  Keep the original [QUESTION] as it is.
4.  Present the final result with the original question followed by the "ELONGATED OPTIONS" labeled A, B, C, and D.

### EXAMPLE
This is an example of how you should perform the task.

[INPUT]
QUESTION: What is the primary source of energy for the Earth's climate system?
OPTIONS:
A. The Sun
B. The Moon
C. Geothermal heat
D. Fossil fuels

[YOUR wzxhzdk:4]
QUESTION: What is the primary source of energy for the Earth's climate system?
ELONGATED OPTIONS:
A. The Sun, which radiates an immense amount of solar energy that drives weather patterns and ocean currents.
B. The Moon, whose gravitational pull is primarily responsible for the ocean's tides rather than providing energy.
C. Geothermal heat, which is the heat energy originating from the Earth's own core and mantle.
D. Fossil fuels, which are carbon-based energy sources from ancient organic matter, burned by humans for energy.

### TASK
Now, apply the same logic to the following input.

[INPUT]
QUESTION: {question}
OPTIONS:
A. {option_a}
B. {option_b}
C. {option_c}
D. {option_d}
"""


# 3. LLM 출력 파싱 함수 (이전과 동일)
# ---------------------------------------------------
def parse_elongated_options(generated_text: str):
    """LLM이 생성한 전체 텍스트에서 길어진 옵션 A, B, C, D를 추출합니다."""
    try:
        content = generated_text.split("ELONGATED OPTIONS:")[1]
    except IndexError:
        print("경고: 'ELONGATED OPTIONS:' 마커를 찾지 못했습니다.")
        content = generated_text

    option_a = re.search(r"A\.\s*(.*?)(?=\s*B\.|\Z)", content, re.DOTALL)
    option_b = re.search(r"B\.\s*(.*?)(?=\s*C\.|\Z)", content, re.DOTALL)
    option_c = re.search(r"C\.\s*(.*?)(?=\s*D\.|\Z)", content, re.DOTALL)
    option_d = re.search(r"D\.\s*(.*)", content, re.DOTALL)

    return {
        'A': option_a.group(1).strip() if option_a else "",
        'B': option_b.group(1).strip() if option_b else "",
        'C': option_c.group(1).strip() if option_c else "",
        'D': option_d.group(1).strip() if option_d else ""
    }

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer 로딩 중...
Pipeline 설정 중... (시간이 소요될 수 있습니다)




Loading checkpoint shards: 100%|██████████| 9/9 [00:04<00:00,  2.02it/s]
Some parameters are on the meta device because they were offloaded to the cpu.
Device set to use cuda:0


모델 및 Pipeline 준비 완료.


In [ ]:
# 4. 메인 처리 로직 (⭐️⭐️⭐️ 수정된 부분 ⭐️⭐️⭐️)
# ---------------------------------------------------
input_csv_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/dataset/visual7w/train.csv'
output_csv_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/llm/train_elongated_10000_samples_live.csv'

NUM_SAMPLES = 10000
RANDOM_SEED = 42

print(f"'{input_csv_path}' 파일 읽기 시작...")
df = pd.read_csv(input_csv_path)

print(f"전체 데이터에서 {NUM_SAMPLES}개의 행을 랜덤으로 샘플링합니다. (Random Seed: {RANDOM_SEED})")
if len(df) > NUM_SAMPLES:
    df_to_process = df.sample(n=NUM_SAMPLES, random_state=RANDOM_SEED)
else:
    df_to_process = df

# -- 결과 파일 초기화 --
# 스크립트를 실행할 때마다 깨끗한 새 파일에서 시작하도록,
# 반복문 시작 전에 파일을 생성하고 헤더(컬럼명)를 먼저 써줍니다.
header_df = pd.DataFrame(columns=['ID', 'img_path', 'Question', 'A', 'B', 'C', 'D', 'answer'])
header_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

print("데이터 처리 및 실시간 저장 시작...")
for index, row in tqdm(df_to_process.iterrows(), total=len(df_to_process)):
    current_prompt = prompt_template.format(
        question=row['Question'],
        option_a=row['A'],
        option_b=row['B'],
        option_c=row['C'],
        option_d=row['D']
    )

    sequences = pipeline(
        current_prompt,
        max_length=1024,
        do_sample=True,
        top_k=10,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_text = sequences[0]['generated_text'].replace(current_prompt, "")
    elongated_options = parse_elongated_options(generated_text)

    # 처리된 한 줄의 데이터를 딕셔너리로 만듭니다.
    new_row = {
        'ID': row['ID'],
        'img_path': row['img_path'],
        'Question': row['Question'],
        'A': elongated_options['A'],
        'B': elongated_options['B'],
        'C': elongated_options['C'],
        'D': elongated_options['D'],
        'answer': row['answer']
    }

    # -- 실시간 저장 로직 --
    # 한 줄짜리 데이터프레임으로 변환합니다.
    output_df = pd.DataFrame([new_row])
    # mode='a' (append) 옵션으로 파일의 끝에 한 줄을 추가합니다.
    # header=False로 설정하여 컬럼명이 계속 추가되는 것을 방지합니다.
    output_df.to_csv(output_csv_path, mode='a', header=False, index=False, encoding='utf-8-sig')


# 5. 결과 저장 (⭐️⭐️⭐️ 수정된 부분 ⭐️⭐️⭐️)
# ---------------------------------------------------
# 모든 데이터가 반복문 안에서 실시간으로 저장되었으므로,
# 마지막에 별도로 저장하는 과정은 필요 없습니다.
print("모든 작업이 성공적으로 완료되었습니다!")
print(f"결과가 실시간으로 저장된 파일: {output_csv_path}")

'/data/2_data_server/cv-07/dice/2025_samsung_challenge/dataset/visual7w/train.csv' 파일 읽기 시작...
전체 데이터에서 10000개의 행을 랜덤으로 샘플링합니다. (Random Seed: 42)
데이터 처리 및 실시간 저장 시작...


  0%|          | 0/10000 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
  0%|          | 0/10000 [00:02<?, ?it/s]


KeyboardInterrupt: 

: 